# 04. Анализ F30

`F30` по справочнику АВТ — расход фракции 290–350 °C с установки. Анализ разделяет уровневые корреляции, совместные изменения, лаги и устойчивость связей по годам. Корреляции не интерпретируются как причинный эффект.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
EDA_DIR = HERE if HERE.name == 'eda' else HERE / 'eda'
ROOT = EDA_DIR.parent
DATA_DIR, DOCS_DIR = ROOT / 'data', ROOT / 'docs'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))
from eda_utils import load_telemetry
from forensics_utils import parse_lims

avt = load_telemetry(DATA_DIR / 'avt_tags.csv')
hydro = load_telemetry(DATA_DIR / '242000_tags.csv')
lims = parse_lims(next(DOCS_DIR.glob('ЛИМС*.xlsx')))
print(avt.shape, hydro.shape, lims.shape)

(189217, 72) (189217, 27) (36800, 9)


## Распределение и подозрительные режимы

In [2]:
f30 = avt[['date','F30']].copy()
profile = pd.DataFrame([{
    'n': len(f30), 'start': f30.date.min(), 'end': f30.date.max(),
    'mean': f30.F30.mean(), 'std': f30.F30.std(), 'min': f30.F30.min(),
    'p01': f30.F30.quantile(.01), 'p05': f30.F30.quantile(.05),
    'median': f30.F30.median(), 'p95': f30.F30.quantile(.95),
    'p99': f30.F30.quantile(.99), 'max': f30.F30.max(),
    'negative_count': int(f30.F30.lt(0).sum()),
    'below_10_count': int(f30.F30.lt(10).sum()),
    'exact_251_count': int(f30.F30.eq(251).sum()),
    'exact_307_count': int(f30.F30.eq(307).sum()),
}])
display(profile)
profile.to_csv(ARTIFACTS / 'f30_profile.csv', index=False)
px.histogram(f30, x='F30', nbins=150, log_y=True, title='F30: распределение, логарифмическая шкала частоты').show()

,n,start,end,mean,std,min,p01,p05,median,p95,p99,max,negative_count,below_10_count,exact_251_count,exact_307_count
0,189217,2023-01-01,2026-08-07,124.007531,29.464454,-1.78614,-0.580889,95.073465,127.916191,153.127997,170.613348,307.0,5081,6815,859,15


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
group = f30.F30.ne(f30.F30.shift()).cumsum()
episodes = f30.groupby(group).agg(start=('date','min'), end=('date','max'), value=('F30','first'), rows=('F30','size'))
episodes['duration_hours'] = (episodes.end - episodes.start).dt.total_seconds()/3600 + 1/6
episodes = episodes.query('rows >= 6').sort_values('duration_hours', ascending=False).reset_index(drop=True)
display(episodes.head(20))
episodes.to_csv(ARTIFACTS / 'f30_constant_episodes.csv', index=False)

monthly = f30.set_index('date').F30.resample('MS').agg(['mean','median','std','min','max']).reset_index()
monthly.to_csv(ARTIFACTS / 'f30_monthly_profile.csv', index=False)
px.line(monthly, x='date', y=['median','mean'], title='F30: месячные mean и median').show()

Физический расход не должен быть отрицательным. Поэтому значения ниже 10, а также повторяющиеся уровни 251/307 рассматриваются как останов, невалидный режим или кодовое значение. Это диагностическая маска, не паспортный operating limit.

## W70 — контрольный сигнал той же фракции

In [ ]:
normal = avt[avt.F30.between(10, 200) & avt.W70.gt(0)].copy()
normal['W70_to_F30'] = normal.W70 / normal.F30
ratio_profile = normal.W70_to_F30.describe(percentiles=[.01,.05,.5,.95,.99]).rename('W70/F30').to_frame()
display(ratio_profile)

bounds = normal[['F30','W70']].quantile([.01,.99])
fit_data = normal[normal.F30.between(bounds.loc[.01,'F30'], bounds.loc[.99,'F30']) &
                  normal.W70.between(bounds.loc[.01,'W70'], bounds.loc[.99,'W70'])]
slope, intercept = np.polyfit(fit_data.F30, fit_data.W70, 1)
r2 = np.corrcoef(fit_data.F30, fit_data.W70)[0,1]**2
fit_summary = pd.DataFrame([{'slope_W70_per_F30': slope, 'intercept': intercept, 'R2': r2,
                             'median_W70_F30_ratio': normal.W70_to_F30.median()}])
display(fit_summary)
fit_summary.to_csv(ARTIFACTS / 'f30_w70_relationship.csv', index=False)

sample = fit_data.iloc[::20]
fig = px.scatter(sample, x='F30', y='W70', opacity=.25, title='F30 и W70 в рабочем диапазоне')
line_x = np.array([fit_data.F30.min(), fit_data.F30.max()])
fig.add_scatter(x=line_x, y=slope*line_x+intercept, mode='lines', name='линейная аппроксимация')
fig.show()

После исключения нерабочих режимов Spearman(F30, W70) ≈ 0.998. Линейная зависимость W70 ≈ 0.773·F30 + 1.04 имеет R² ≈ 0.996; медиана W70/F30 ≈ 0.781. Это согласуется с гипотезой, что W70 — массовый, а F30 — объёмный расход одной фракции. Без единиц это остаётся сильной, но всё же гипотезой.

## Связи с другими параметрами АВТ

In [ ]:
process = normal.drop(columns='W70_to_F30').set_index('date')
level_corr = process.corr(method='spearman')['F30'].drop('F30')
change_corr = process.diff().corr(method='spearman')['F30'].drop('F30')
relationships = pd.DataFrame({'level_spearman': level_corr, 'change_spearman': change_corr}).reset_index(names='tag')

year_rows = []
for tag in relationships.tag:
    vals = {}
    for year, frame in normal.groupby(normal.date.dt.year):
        vals[f'spearman_{year}'] = frame[['F30',tag]].corr(method='spearman').iloc[0,1]
    year_rows.append({'tag':tag, **vals})
year_corr = pd.DataFrame(year_rows)
relationships = relationships.merge(year_corr, on='tag')
year_cols = [c for c in relationships if c.startswith('spearman_')]
relationships['year_min'] = relationships[year_cols].min(axis=1)
relationships['year_max'] = relationships[year_cols].max(axis=1)
relationships['year_range'] = relationships.year_max - relationships.year_min
relationships['abs_level'] = relationships.level_spearman.abs()
relationships = relationships.sort_values('abs_level', ascending=False)
display(relationships.head(25))
relationships.to_csv(ARTIFACTS / 'f30_avt_relationships.csv', index=False)
px.bar(relationships.head(20), x='level_spearman', y='tag', orientation='h',
       color='change_spearman', title='F30: связи с тегами АВТ').show()

Кроме W70, сильнее всего с уровнем F30 связаны F65, F31, F16, F29, F59 и F60. Но для приращений ΔF30 большинство этих связей почти исчезают. Это признак общего режима/производительности, а не немедленного воздействия F30. Исключение — F32: Spearman уровней около 0.41 и приращений около 0.42, то есть соседний поток меняется вместе с F30 и на коротком масштабе.

## Инерционность F30

In [ ]:
valid_f30 = avt.F30.where(avt.F30.between(10,200))
autocorr = pd.DataFrame([
    {'lag':'10 min','steps':1}, {'lag':'1 h','steps':6}, {'lag':'6 h','steps':36},
    {'lag':'24 h','steps':144}, {'lag':'72 h','steps':432}, {'lag':'7 d','steps':1008},
])
autocorr['spearman'] = [valid_f30.corr(valid_f30.shift(s), method='spearman') for s in autocorr.steps]
display(autocorr)
autocorr.to_csv(ARTIFACTS / 'f30_autocorrelation.csv', index=False)
px.bar(autocorr, x='lag', y='spearman', title='F30: автокорреляция').show()

F30 очень инерционен: Spearman около 0.99 на 1 час, 0.96 на 6 часов и 0.86 на 24 часа. Из-за этого соседние лаги дают почти одинаковые корреляции. Пик lagged correlation нельзя объявлять временем переноса без дополнительного анализа переходных режимов.

## Связи с телеметрией 24-2000

In [ ]:
f_hour = avt.set_index('date').F30.resample('1h').median().where(lambda s:s.between(10,200))
h_hour = hydro.set_index('date').resample('1h').median()
cross_rows = []
for lag_h in [0,1,2,4,6,8,12,24,48,72]:
    joined = h_hour.join(f_hour.shift(lag_h).rename('F30'))
    level = joined.corr(method='spearman').F30.drop('F30')
    change = joined.diff().corr(method='spearman').F30.drop('F30')
    for tag in level.index:
        cross_rows.append({'tag_24_2000':tag, 'lag_hours':lag_h,
                           'level_spearman':level[tag], 'change_spearman':change[tag]})
cross = pd.DataFrame(cross_rows)
cross['abs_level'] = cross.level_spearman.abs()
cross['abs_change'] = cross.change_spearman.abs()
best_level = cross.loc[cross.groupby('tag_24_2000').abs_level.idxmax()].sort_values('abs_level', ascending=False)
best_change = cross.loc[cross.groupby('tag_24_2000').abs_change.idxmax()].sort_values('abs_change', ascending=False)
display(best_level.head(15), best_change.head(15))
cross.to_csv(ARTIFACTS / 'f30_cross_installation_lags.csv', index=False)

Уровень F30 умеренно связан с F26, F9, W10, F1, F19 и F17 установки 24-2000 (|ρ| около 0.43–0.46), преимущественно при нулевом лаге. Но связи приращений слабые: максимум около 0.11. Это снова похоже на общий производственный режим, а не на быстрое прямое влияние.

## F30 и лабораторное качество

In [ ]:
source_f30 = avt.loc[avt.F30.between(10,200), ['date','F30']].sort_values('date')
lag_rows = []
for (installation, point, target), lab in lims.groupby(['installation','sampling_point','quality_parameter']):
    if len(lab) < 30:
        continue
    lab = lab[['timestamp','value']].sort_values('timestamp')
    for lag_h in [0,1,2,4,6,8,12,24,48,72]:
        shifted = source_f30.copy()
        shifted['source_timestamp'] = shifted.date
        shifted['date'] = shifted.date + pd.Timedelta(hours=lag_h)
        joined = pd.merge_asof(lab, shifted, left_on='timestamp', right_on='date',
                               direction='nearest', tolerance=pd.Timedelta('5min')).dropna()
        if len(joined) >= 30:
            lag_rows.append({'installation':installation,'sampling_point':point,'target':target,
                'lag_hours':lag_h,'n_pairs':len(joined),
                'spearman':joined[['value','F30']].corr(method='spearman').iloc[0,1],
                'pearson':joined[['value','F30']].corr().iloc[0,1]})
quality_lags = pd.DataFrame(lag_rows)
quality_lags['abs_spearman'] = quality_lags.spearman.abs()
hydro_product = quality_lags.query("installation == 'Гидроочистка' and sampling_point == '2'")
best_quality = hydro_product.loc[hydro_product.groupby('target').abs_spearman.idxmax()].sort_values('abs_spearman', ascending=False)
display(best_quality)
quality_lags.to_csv(ARTIFACTS / 'f30_lims_lag_associations.csv', index=False)
px.imshow(hydro_product.pivot(index='target', columns='lag_hours', values='spearman'),
          aspect='auto', zmin=-.5, zmax=.5, color_continuous_scale='RdBu_r',
          title='F30(t−lag) и LIMS товарного дизеля(t): Spearman').show()

In [ ]:
stability_rows = []
for target in ['FlashPoint','Mg.Sulfur','IBP.T','90%.T','95%.T']:
    lab = lims.query("installation == 'Гидроочистка' and sampling_point == '2' and quality_parameter == @target")[['timestamp','value']].sort_values('timestamp')
    joined = pd.merge_asof(lab, source_f30, left_on='timestamp', right_on='date',
                           direction='nearest', tolerance=pd.Timedelta('5min')).dropna()
    for year, frame in joined.groupby(joined.timestamp.dt.year):
        stability_rows.append({'target':target,'year':year,'n':len(frame),
            'spearman':frame[['value','F30']].corr(method='spearman').iloc[0,1]})
quality_stability = pd.DataFrame(stability_rows)
display(quality_stability.pivot(index='target',columns='year',values='spearman'))
quality_stability.to_csv(ARTIFACTS / 'f30_quality_year_stability.csv', index=False)

Для серы связь практически отсутствует: лучший |Spearman| около 0.03. F30 отдельно не является полезным предиктором серы.

Самая заметная связь с товарным качеством — отрицательная с FlashPoint при lag=0 (ρ≈−0.31, n≈1509), но она ослабевает от −0.43 в 2023 до −0.11 в 2025. Связи с IBP/T90/T95 также меняют знак или силу по годам. Это режимные корреляции, которые нельзя переносить в управляющую рекомендацию без multivariate model и временной проверки.

## Практические выводы

1. F30 — важный индикатор загрузки/выхода фракции 290–350 °C, но не самостоятельный рычаг качества.
2. W70 можно использовать как redundancy check. Большое отклонение отношения W70/F30 от рабочего диапазона примерно 0.764–0.792 требует флага качества данных. Это observed range, не технологический limit.
3. Значения F30 < 10, отрицательные значения и плато 251/307 нужно отделять как stop/invalid/encoded regimes.
4. Для модели полезны F30, W70, F32 и отношения F30/(F30+F32), W70/F30, а также rolling mean/slope и time-since-change.
5. Сильные уровневые корреляции F30 с производительностью не означают прямое влияние: корреляции приращений значительно слабее.
6. Для серы F30 имеет смысл только внутри многомерной lagged-модели совместно с параметрами гидроочистки.